============================================================================
pPred — External Validation Script
============================================================================
Evaluates the frozen pPred models (ExtraTrees classifier + Stacking
regressor) on an independent external validation set built from:
  1. ChEMBL refresh  (new IC50 records for the 3 training targets)
  2. BindingDB       (PD-L1, UniProt Q9NZQ7)
  3. PubChem         (biochemical PD-1/PD-L1 IC50 assays)

LOCKED A PRIORI :
  - Active = IC50 <= 1000 nM (identical to training)
  - pIC50 = -log10(IC50_nM * 1e-9), capped at 1e8 nM
  - IC50 measurements only (no Ki/Kd)
  - No retraining, no refitting, no threshold adjustment


#NOTE ON DEDUP: the original run also excluded ChEMBL records by activity_id/molecule_chembl_id using the cached training fetch. This version relies on structural dedup against the 4,398 training SMILES, which removes the same compounds (same structure = same standardized SMILES, regardless of ChEMBL record ID).
============================================================================


============================================================================
SECTION 0 — Setup: install packages, download frozen models from GitHub
============================================================================


In [1]:
import subprocess, sys

def _pip(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

_pip("rdkit")
_pip("scikit-learn>=1.9")   # match training version to avoid unpickle warnings
_pip("xgboost")
_pip("lightgbm")

import os, time, json, pickle, warnings
import numpy as np
import pandas as pd
import requests
from io import StringIO

warnings.filterwarnings("ignore")
from tqdm.auto import tqdm

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, QED, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
RDLogger.DisableLog("rdApp.*")

OUT_DIR = "/content/ppred_output"
os.makedirs(OUT_DIR, exist_ok=True)



### Download frozen models from the GitHub repo


In [2]:
GITHUB_RAW = "https://raw.githubusercontent.com/Ericaakanko/pPred_Project/main"
MODEL_FILES = [
    "classifier_model.pkl",
    "regressor_model.pkl",
    "feature_selector.pkl",
    "train_smiles.pkl",
]

os.makedirs("/content/models", exist_ok=True)
for fname in MODEL_FILES:
    dest = f"/content/models/{fname}"
    if not os.path.exists(dest):
        url = f"{GITHUB_RAW}/models/{fname}"
        print(f"Downloading {fname} ...")
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        with open(dest, "wb") as f:
            f.write(r.content)
print("All model files ready.\n")



All model files ready.



============================================================================
SECTION 1 — Configuration (locked a priori)
============================================================================


In [3]:
RADIUS, N_BITS = 2, 2048
ACTIVE_THRESHOLD = 1000  # nM
N_BOOTSTRAP = 2000       # reduce to 500 for a faster run
RANDOM_SEED = 42

TARGET_IDS = {
    "CHEMBL3580522": "PD-L1 single protein",
    "CHEMBL4523993": "PD-1/PD-L1 protein complex",
    "CHEMBL6066575": "PD-L1/2 protein family",
}
BDB_TARGETS = {"Q9NZQ7": "PD-L1 (CD274)", "P21781": "PD-1 (PDCD1)"}
VALID_PC_TARGETS = {"Q9NZQ7", "P21781"}

DESC_COLUMNS = [
    "MW", "LogP", "TPSA", "NumHDonors", "NumHAcceptors",
    "NumRotatableBonds", "NumAromaticRings", "NumAliphaticRings",
    "FractionCSP3", "NumHeavyAtoms", "NumRings", "BalabanJ",
    "BertzCT", "MolMR", "LabuteASA", "NumValenceElectrons", "QED",
]


Identical to the training pipeline: largest fragment, neutralize, canonicalize.


In [4]:
def standardize_smiles(smi):
    """Identical to the training pipeline: largest fragment, neutralize, canonicalize."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    mol = rdMolStandardize.LargestFragmentChooser().choose(mol)
    mol = rdMolStandardize.Uncharger().uncharge(mol)
    return Chem.MolToSmiles(mol)

def ic50_to_pic50(ic50_nM):
    return -np.log10(min(ic50_nM, 1e8) * 1e-9)

print("Configuration locked:")
print(f"  Active threshold : IC50 <= {ACTIVE_THRESHOLD} nM")
print(f"  Endpoint         : IC50 only")
print(f"  Bootstrap        : {N_BOOTSTRAP} resamples\n")



Configuration locked:
  Active threshold : IC50 <= 1000 nM
  Endpoint         : IC50 only
  Bootstrap        : 2000 resamples



============================================================================
SECTION 2 — Data collection (3 sources, IC50/nM only)
============================================================================


In [5]:
HEADERS = {"User-Agent": "Mozilla/5.0 (Colab research script)"}

def robust_get(url, max_retries=5, timeout=90, **kw):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, timeout=timeout, **kw)
            if r.status_code == 200:
                return r
            if r.status_code == 429:          # rate limited — back off hard
                print(f"    429 rate-limited, waiting 60s ...")
                time.sleep(60)
                continue
        except Exception:
            pass
        time.sleep(2 ** attempt)
    return None



### 2a) ChEMBL refresh


In [6]:
print("=" * 60)
print("SECTION 2a — ChEMBL refresh")
print("=" * 60)
BASE = "https://www.ebi.ac.uk/chembl/api/data"

act_records = []
for tid, tname in TARGET_IDS.items():
    offset, n_fetched = 0, 0
    while True:
        d = None
        url = (f"{BASE}/activity.json?target_chembl_id={tid}"
               f"&standard_type=IC50&standard_units=nM&limit=1000&offset={offset}")
        r = robust_get(url, timeout=120)
        if r is None:
            print(f"  WARNING: {tid} fetch failed at offset {offset}, skipping remainder")
            break
        d = r.json()
        acts = d.get("activities", [])
        if not acts:
            break
        act_records.extend(acts)
        n_fetched += len(acts)
        offset += len(acts)
        if len(acts) < 1000:
            break
    print(f"  {tid} ({tname}): {n_fetched} IC50/nM activities")

df_act = pd.DataFrame(act_records)
print(f"  Total fetched: {len(df_act)} records")

# Fetch canonical SMILES for unique molecules
new_mol_ids = sorted(df_act["molecule_chembl_id"].unique())
print(f"  Fetching SMILES for {len(new_mol_ids)} molecules ...")
mol_smiles = {}
for i in tqdm(range(0, len(new_mol_ids), 50), desc="  ChEMBL molecules"):
    batch = new_mol_ids[i:i + 50]
    r = robust_get(f"{BASE}/molecule.json?molecule_chembl_id__in={','.join(batch)}&limit=100")
    if r is None:
        continue
    for mol in r.json().get("molecules", []):
        smi = (mol.get("molecule_structures") or {}).get("canonical_smiles")
        if smi:
            mol_smiles[mol["molecule_chembl_id"]] = smi
    time.sleep(0.3)

df_chembl = df_act[df_act["molecule_chembl_id"].isin(mol_smiles)].copy()
df_chembl["canonical_smiles"] = df_chembl["molecule_chembl_id"].map(mol_smiles)
df_chembl["standard_value"] = pd.to_numeric(df_chembl["standard_value"], errors="coerce")
df_chembl = df_chembl.dropna(subset=["canonical_smiles", "standard_value"])
df_chembl = df_chembl[df_chembl["standard_value"] > 0]

df_ch_ext = pd.DataFrame({
    "source": "ChEMBL_refresh",
    "source_id": df_chembl["activity_id"].astype(str),
    "smiles_raw": df_chembl["canonical_smiles"],
    "ic50_nM": df_chembl["standard_value"],
    "relation": df_chembl["relation"].fillna("="),
    "target_info": df_chembl["target_chembl_id"],
    "document": df_chembl["document_chembl_id"],
})
print(f"  ChEMBL refresh records: {len(df_ch_ext)}\n")



SECTION 2a — ChEMBL refresh
  CHEMBL3580522 (PD-L1 single protein): 1772 IC50/nM activities
  CHEMBL4523993 (PD-1/PD-L1 protein complex): 5880 IC50/nM activities
  CHEMBL6066575 (PD-L1/2 protein family): 432 IC50/nM activities
  Total fetched: 8084 records
  Fetching SMILES for 5846 molecules ...


  ChEMBL molecules:   0%|          | 0/117 [00:00<?, ?it/s]

  ChEMBL refresh records: 8075



### 2b) BindingDB


In [7]:
print("=" * 60)
print("SECTION 2b — BindingDB")
print("=" * 60)
bdb_frames = []
for uniprot, tname in BDB_TARGETS.items():
    url = (f"https://bindingdb.org/rest/getLigandsByUniprots"
           f"?uniprot={uniprot}&cutoff=100000&response=application/json")
    r = robust_get(url, timeout=300)
    if r is None:
        print(f"  {tname}: unavailable (skipping)")
        continue
    try:
        affinities = r.json()["getLindsByUniprotsResponse"]["affinities"]
        df_b = pd.DataFrame(affinities)
        df_b["bdb_target"] = tname
        bdb_frames.append(df_b)
        print(f"  {tname}: {len(df_b)} records")
    except Exception as e:
        print(f"  {tname}: parse failed ({e})")
    time.sleep(3)

if bdb_frames:
    df_bdb = pd.concat(bdb_frames, ignore_index=True)
    df_bdb["ic50_nM"] = pd.to_numeric(df_bdb["affinity"], errors="coerce")
    str_vals = df_bdb["affinity"].astype(str).str.strip()
    df_bdb["relation"] = np.where(str_vals.str.startswith("<"), "<",
                          np.where(str_vals.str.startswith(">"), ">", "="))
    df_bdb["ic50_nM"] = df_bdb["ic50_nM"].fillna(
        pd.to_numeric(str_vals.str.lstrip("<>"), errors="coerce"))
    df_bdb = df_bdb[df_bdb["affinity_type"] == "IC50"]
    df_bdb = df_bdb[df_bdb["ic50_nM"] > 0]
    df_bdb_ext = pd.DataFrame({
        "source": "BindingDB",
        "source_id": "BDB_" + df_bdb["monomerid"].astype(str),
        "smiles_raw": df_bdb["smile"],
        "ic50_nM": df_bdb["ic50_nM"],
        "relation": df_bdb["relation"],
        "target_info": df_bdb["query"],
        "document": df_bdb["doi"],
    })
    print(f"  BindingDB IC50 records: {len(df_bdb_ext)}")
else:
    df_bdb_ext = pd.DataFrame(columns=["source", "source_id", "smiles_raw",
                                       "ic50_nM", "relation", "target_info", "document"])
    print("  BindingDB unavailable — continuing without it")



SECTION 2b — BindingDB
  PD-L1 (CD274): 10563 records
  PD-1 (PDCD1): 0 records
  BindingDB IC50 records: 9929


### 2c) PubChem


In [8]:
print("=" * 60)
print("SECTION 2c — PubChem")
print("=" * 60)
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
PUG = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

def esearch(term, retmax=200):
    r = robust_get(f"{EUTILS}/esearch.fcgi", params={
        "db": "pcassay", "term": term, "retmax": retmax, "retmode": "json"})
    return r.json().get("esearchresult", {}).get("idlist", []) if r else []

def esummary(aids):
    out = {}
    for i in range(0, len(aids), 50):
        batch = aids[i:i + 50]
        r = robust_get(f"{EUTILS}/esummary.fcgi", params={
            "db": "pcassay", "id": ",".join(batch), "retmode": "json"})
        if r is None:
            continue
        d = r.json().get("result", {})
        for aid in batch:
            if aid in d:
                out[aid] = d[aid]
        time.sleep(0.4)
    return out

aids = sorted(set(esearch("PD-L1 AND IC50")) | set(esearch("PD-1 AND IC50")))
summaries = esummary(aids)
print(f"  Candidate AIDs: {len(aids)}")

# Keep only strictly biochemical, non-cell-based PD-1/PD-L1 assays
biochemical = []
for aid, s in summaries.items():
    text = (str(s.get("assayname", "")) + " " + str(s.get("assaydescription", ""))).lower()
    has_target = any(k in text for k in [
        "pd-l1", "pdl1", "pd-1", "pd1 ", "pd1/", "pd1/pdl1",
        "programmed cell death 1", "programmed death", "cd274", "pdcd1"])
    is_cell = any(k in text for k in [
        "cell-based", "cellular", "in cells", "cell viability", "reporter gene in cell",
        "293t", "hek293", "cho-k1", "jurkat", "cell line", "pbmc", "protac",
        "cell surface expression", "dimerization in"])
    is_bio = any(k in text for k in [
        "htrf", "tr-fret", "fret", "fluorescence polarization", "alpha screen",
        "alphascreen", "elisa", "homogeneous time-resolved", "homogenous time-resolved",
        "competitive binding", "fluorescence assay", "surface plasmon resonance",
        "differential scanning fluorimetry", "in vitro", "binding assay",
        "protein-protein interaction"])
    if has_target and is_bio and not is_cell:
        biochemical.append(aid)
print(f"  Biochemical PD-1/PD-L1 AIDs: {len(biochemical)}")

rows = []
failed_aids = 0
for i, aid in enumerate(biochemical):
    r = robust_get(f"{PUG}/assay/aid/{aid}/concise/JSON", timeout=120)
    if r is None:
        failed_aids += 1
        continue
    try:
        tbl = r.json()["Table"]
        cols = tbl["Columns"]["Column"]
        for row in tbl.get("Row", []):
            rec = dict(zip(cols, row["Cell"]))
            rec["AID"] = aid
            rows.append(rec)
    except Exception:
        failed_aids += 1
        continue
    if (i + 1) % 20 == 0:
        print(f"    {i+1}/{len(biochemical)} AIDs, {len(rows)} rows")
    time.sleep(1.0)
if failed_aids:
    print(f"  WARNING: {failed_aids}/{len(biochemical)} AIDs failed to fetch "
          f"(likely rate limiting) — PubChem coverage reduced.")

df_pc = pd.DataFrame(rows)
df_pc_ext = pd.DataFrame(columns=["source", "source_id", "smiles_raw",
                                  "ic50_nM", "relation", "target_info", "document"])
if len(df_pc):
    val_col = [c for c in df_pc.columns if c.startswith("Activity Value")][0]
    df_pc["val_uM"] = pd.to_numeric(df_pc[val_col], errors="coerce")
    df_pc["tgt"] = df_pc["Target Accession"].fillna("").astype(str).str.strip()
    mask = (df_pc["tgt"].isin(VALID_PC_TARGETS)
            & df_pc["Activity Name"].str.strip().str.lower().eq("ic50")
            & df_pc["val_uM"].notna() & (df_pc["val_uM"] > 0)
            & df_pc["Activity Outcome"].isin(["Active", "Inactive"]))
    df_pc_ic50 = df_pc[mask].copy()
    df_pc_ic50["ic50_nM"] = df_pc_ic50["val_uM"] * 1000.0  # uM -> nM

    # Map CIDs to SMILES
    cids = sorted(df_pc_ic50["CID"].astype(str).unique())
    print(f"  Mapping {len(cids)} CIDs to SMILES ...")
    cid_smiles = {}
    failed_batches = 0
    for i in range(0, len(cids), 100):
        batch = cids[i:i + 100]
        r = robust_get(f"{PUG}/compound/cid/{','.join(batch)}/property/SMILES/JSON")
        if r is None:
            failed_batches += 1
            continue
        for prop in r.json().get("PropertyTable", {}).get("Properties", []):
            if prop.get("SMILES"):
                cid_smiles[str(prop.get("CID"))] = prop["SMILES"]
        time.sleep(1.0)
    if failed_batches:
        print(f"  WARNING: {failed_batches} SMILES batches failed (rate limit?). "
              f"Mapped {len(cid_smiles)}/{len(cids)} CIDs.")
    if len(cid_smiles) == 0:
        print("  WARNING: No PubChem SMILES could be retrieved — PubChem data will be "
              "EXCLUDED from the external set. Wait ~5 min and re-run, or proceed "
              "with ChEMBL + BindingDB only.")

    df_pc_ext = pd.DataFrame({
        "source": "PubChem",
        "source_id": "AID" + df_pc_ic50["AID"].astype(str) + "_CID" + df_pc_ic50["CID"].astype(str),
        "smiles_raw": df_pc_ic50["CID"].astype(str).map(cid_smiles),
        "ic50_nM": df_pc_ic50["ic50_nM"],
        "relation": "=",
        "target_info": df_pc_ic50["tgt"].map({"Q9NZQ7": "PD-L1 (CD274)",
                                              "P21781": "PD-1 (PDCD1)"}),
        "document": "AID" + df_pc_ic50["AID"].astype(str),
    })
    df_pc_ext = df_pc_ext[df_pc_ext["smiles_raw"].notna()]
    print(f"  PubChem IC50 records: {len(df_pc_ext)}")



SECTION 2c — PubChem
    429 rate-limited, waiting 60s ...
  Candidate AIDs: 265
  Biochemical PD-1/PD-L1 AIDs: 83
    20/83 AIDs, 2495 rows
    40/83 AIDs, 5001 rows
    60/83 AIDs, 6674 rows
    80/83 AIDs, 8931 rows
  Mapping 4796 CIDs to SMILES ...
  PubChem IC50 records: 5768


============================================================================
SECTION 3 — Standardize + 3-layer dedup -> external validation set
============================================================================


In [9]:
print("=" * 60)
print("SECTION 3 — Standardization + deduplication")
print("=" * 60)
pool = pd.concat([df_ch_ext, df_bdb_ext, df_pc_ext], ignore_index=True)
print(f"Pooled external records: {len(pool)}")
print(f"  {pool['source'].value_counts().to_dict()}")

pool["std_smiles"] = [standardize_smiles(s) if s else None for s in pool["smiles_raw"]]
pool = pool[pool["std_smiles"].notna()]
print(f"After standardization: {len(pool)}")

# Layer b: structural dedup vs training (4,398 SMILES)
with open("/content/models/train_smiles.pkl", "rb") as f:
    train_smiles = pickle.load(f)
TRAIN_SET = set(train_smiles)
n1 = len(pool)
pool = pool[~pool["std_smiles"].isin(TRAIN_SET)].copy()
print(f"After structural dedup vs training: {n1} -> {len(pool)} ({n1 - len(pool)} removed)")

# Layer c: cross-source dedup, median IC50
pool["relation"] = pool["relation"].fillna("=")
ext = pool.groupby("std_smiles").agg(
    ic50_nM=("ic50_nM", "median"),
    sources=("source", lambda x: ";".join(sorted(set(x)))),
    n_measurements=("ic50_nM", "size"),
    relation=("relation", "first"),
    target_info=("target_info", "first"),
    document=("document", "first"),
).reset_index()
print(f"After cross-source dedup: {len(ext)} unique external compounds")

ext["pIC50"] = ext["ic50_nM"].apply(ic50_to_pic50)
ext["activity"] = (ext["ic50_nM"] <= ACTIVE_THRESHOLD).astype(int)
ext["class"] = ext["activity"].map({1: "active", 0: "inactive"})
print(f"  Active: {ext['activity'].sum()} ({ext['activity'].mean():.1%}) | "
      f"Inactive: {(1 - ext['activity']).sum()}")

pool.to_csv(f"{OUT_DIR}/external_pooled_records.csv", index=False)
ext.to_csv(f"{OUT_DIR}/external_validation_set.csv", index=False)



SECTION 3 — Standardization + deduplication
Pooled external records: 23772
  {'BindingDB': 9929, 'ChEMBL_refresh': 8075, 'PubChem': 5768}
After standardization: 23772
After structural dedup vs training: 23772 -> 10366 (13406 removed)
After cross-source dedup: 4838 unique external compounds
  Active: 4478 (92.6%) | Inactive: 360


============================================================================
SECTION 4 — Featurize + predict with FROZEN models
============================================================================


In [10]:
print("=" * 60)
print("SECTION 4 — Predictions (frozen models, no retraining)")
print("=" * 60)
with open("/content/models/classifier_model.pkl", "rb") as f:
    clf_model = pickle.load(f)
with open("/content/models/regressor_model.pkl", "rb") as f:
    reg_model = pickle.load(f)
with open("/content/models/feature_selector.pkl", "rb") as f:
    selector = pickle.load(f)
print("Frozen models loaded.")

mols = [Chem.MolFromSmiles(s) for s in ext["std_smiles"]]
valid = [m is not None for m in mols]
ext = ext[valid].reset_index(drop=True)
mols = [m for m in mols if m is not None]
print(f"Valid structures: {len(mols)}")

fps = [AllChem.GetMorganFingerprintAsBitVect(m, RADIUS, nBits=N_BITS) for m in mols]
fp_array = np.zeros((len(fps), N_BITS), dtype=np.int8)
for j, fp in enumerate(fps):
    DataStructs.ConvertToNumpyArray(fp, fp_array[j])

def calc_descriptors(mol):
    return [Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
            rdMolDescriptors.CalcTPSA(mol), Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol), Descriptors.NumRotatableBonds(mol),
            Descriptors.NumAromaticRings(mol), Descriptors.NumAliphaticRings(mol),
            Descriptors.FractionCSP3(mol), Descriptors.HeavyAtomCount(mol),
            Descriptors.RingCount(mol), Descriptors.BalabanJ(mol),
            Descriptors.BertzCT(mol), Descriptors.MolMR(mol),
            Descriptors.LabuteASA(mol), Descriptors.NumValenceElectrons(mol),
            QED.qed(mol)]

desc_array = np.array([calc_descriptors(m) for m in mols], dtype=np.float64)
X_sel = selector.transform(np.hstack([fp_array, desc_array]))
print(f"Features: {N_BITS + 17} -> {X_sel.shape[1]} after VarianceThreshold")

ext["clf_pred"] = clf_model.predict(X_sel)
ext["clf_pred_label"] = np.where(ext["clf_pred"] == 1, "Active", "Inactive")
ext["clf_proba_active"] = clf_model.predict_proba(X_sel)[:, 1]
ext["reg_pred_pIC50"] = reg_model.predict(X_sel)

# Applicability domain (Tanimoto NN vs training fingerprints)
print("Computing applicability domain ...")
train_fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), RADIUS, nBits=N_BITS)
             for s in train_smiles]
nn_sims = np.array([max(DataStructs.BulkTanimotoSimilarity(fp, train_fps)) for fp in
                    tqdm(fps, desc="  AD")])
ext["nn_tanimoto"] = nn_sims
ext["ad_tier"] = np.where(nn_sims >= 0.4, "In-domain",
                  np.where(nn_sims >= 0.3, "Borderline", "Out-of-domain"))
ext.to_csv(f"{OUT_DIR}/external_predictions.csv", index=False)
print(f"AD tiers: {pd.Series(ext['ad_tier']).value_counts().to_dict()}")



SECTION 4 — Predictions (frozen models, no retraining)
Frozen models loaded.
Valid structures: 4838
Features: 2065 -> 2028 after VarianceThreshold
Computing applicability domain ...


  AD:   0%|          | 0/4838 [00:00<?, ?it/s]

AD tiers: {'In-domain': 4749, 'Borderline': 78, 'Out-of-domain': 11}


============================================================================
SECTION 5 — Evaluation (metrics + bootstrap CIs + AD-stratified)
============================================================================


In [11]:
print("=" * 60)
print("SECTION 5 — Evaluation")
print("=" * 60)
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             balanced_accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, cohen_kappa_score,
                             brier_score_loss, confusion_matrix, roc_curve,
                             precision_recall_curve, r2_score, mean_squared_error,
                             mean_absolute_error)
from sklearn.calibration import calibration_curve
from scipy.stats import spearmanr, pearsonr

rng = np.random.default_rng(RANDOM_SEED)
y_true = ext["activity"].values
y_pred = ext["clf_pred"].values
y_proba = ext["clf_proba_active"].values
y_exp = ext["pIC50"].values
y_reg = ext["reg_pred_pIC50"].values

print(f"Confusion matrix: TN={confusion_matrix(y_true, y_pred)[0,0]}, "
      f"FP={confusion_matrix(y_true, y_pred)[0,1]}, "
      f"FN={confusion_matrix(y_true, y_pred)[1,0]}, "
      f"TP={confusion_matrix(y_true, y_pred)[1,1]}")

clf_fns = [
    ("ROC AUC", lambda yt, yp, ypr: roc_auc_score(yt, ypr)),
    ("PR AUC", lambda yt, yp, ypr: average_precision_score(yt, ypr)),
    ("Accuracy", lambda yt, yp, ypr: accuracy_score(yt, yp)),
    ("Balanced accuracy", lambda yt, yp, ypr: balanced_accuracy_score(yt, yp)),
    ("Precision", lambda yt, yp, ypr: precision_score(yt, yp, zero_division=0)),
    ("Recall", lambda yt, yp, ypr: recall_score(yt, yp, zero_division=0)),
    ("F1", lambda yt, yp, ypr: f1_score(yt, yp, zero_division=0)),
    ("MCC", lambda yt, yp, ypr: matthews_corrcoef(yt, yp)),
    ("Cohen's kappa", lambda yt, yp, ypr: cohen_kappa_score(yt, yp)),
    ("Brier score", lambda yt, yp, ypr: brier_score_loss(yt, ypr)),
]
clf_rows = []
for name, fn in clf_fns:
    point = fn(y_true, y_pred, y_proba)
    stats = []
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, len(y_true), len(y_true))
        try:
            stats.append(fn(y_true[idx], y_pred[idx], y_proba[idx]))
        except Exception:
            continue
    lo, hi = np.percentile(stats, 2.5), np.percentile(stats, 97.5)
    clf_rows.append({"Metric": name, "Value": point, "CI_low": lo, "CI_high": hi})
    print(f"  {name}: {point:.3f} [{lo:.3f}, {hi:.3f}]")
pd.DataFrame(clf_rows).to_csv(f"{OUT_DIR}/classification_metrics.csv", index=False)

reg_fns = [
    ("R2", lambda yt, yp: r2_score(yt, yp)),
    ("RMSE", lambda yt, yp: np.sqrt(mean_squared_error(yt, yp))),
    ("MAE", lambda yt, yp: mean_absolute_error(yt, yp)),
    ("Spearman rho", lambda yt, yp: spearmanr(yt, yp).statistic),
    ("Pearson r", lambda yt, yp: pearsonr(yt, yp).statistic),
]
reg_rows = []
for name, fn in reg_fns:
    point = fn(y_exp, y_reg)
    stats = []
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, len(y_exp), len(y_exp))
        try:
            stats.append(fn(y_exp[idx], y_reg[idx]))
        except Exception:
            continue
    lo, hi = np.percentile(stats, 2.5), np.percentile(stats, 97.5)
    reg_rows.append({"Metric": name, "Value": point, "CI_low": lo, "CI_high": hi})
    print(f"  {name}: {point:.3f} [{lo:.3f}, {hi:.3f}]")
pd.DataFrame(reg_rows).to_csv(f"{OUT_DIR}/regression_metrics.csv", index=False)

# AD-stratified
ad_rows = []
for tier in ["In-domain", "Borderline", "Out-of-domain"]:
    sub = ext[ext["ad_tier"] == tier]
    if len(sub) < 10 or sub["activity"].nunique() < 2:
        continue
    yt, yp, ypr = sub["activity"].values, sub["clf_pred"].values, sub["clf_proba_active"].values
    ye, yr = sub["pIC50"].values, sub["reg_pred_pIC50"].values
    ad_rows.append({"AD tier": tier, "n": len(sub),
                    "ROC AUC": roc_auc_score(yt, ypr),
                    "MCC": matthews_corrcoef(yt, yp),
                    "R2": r2_score(ye, yr),
                    "RMSE": np.sqrt(mean_squared_error(ye, yr)),
                    "Spearman": spearmanr(ye, yr).statistic})
ad_df = pd.DataFrame(ad_rows)
ad_df.to_csv(f"{OUT_DIR}/ad_stratified_metrics.csv", index=False)
if len(ad_df):
    print(ad_df.to_string(index=False))

# Per-source
src_rows = []
for src in ["ChEMBL_refresh", "BindingDB", "PubChem"]:
    sub = ext[ext["sources"].str.contains(src)]
    if not len(sub):
        continue
    src_rows.append({
        "Source": src, "n": len(sub), "Active %": 100 * sub["activity"].mean(),
        "ROC AUC": roc_auc_score(sub["activity"], sub["clf_proba_active"]),
        "MCC": matthews_corrcoef(sub["activity"], sub["clf_pred"]),
        "Spearman": spearmanr(sub["pIC50"], sub["reg_pred_pIC50"]).statistic})
src_df = pd.DataFrame(src_rows)
src_df.to_csv(f"{OUT_DIR}/per_source_metrics.csv", index=False)
print(src_df.to_string(index=False))

# Balanced-subset sensitivity (external set is heavily active-enriched)
inact_idx = np.where(y_true == 0)[0]
act_idx = np.where(y_true == 1)[0]
bal = []
for _ in range(100):
    sel = np.concatenate([rng.choice(act_idx, size=len(inact_idx), replace=False), inact_idx])
    bal.append({"ROC AUC": roc_auc_score(y_true[sel], y_proba[sel]),
                "MCC": matthews_corrcoef(y_true[sel], y_pred[sel])})
bal_df = pd.DataFrame(bal)
bal_df.to_csv(f"{OUT_DIR}/balanced_sensitivity.csv", index=False)
print(f"Balanced subset (100x): ROC AUC {bal_df['ROC AUC'].mean():.3f} +/- {bal_df['ROC AUC'].std():.3f} | "
      f"MCC {bal_df['MCC'].mean():.3f} +/- {bal_df['MCC'].std():.3f}")



SECTION 5 — Evaluation
Confusion matrix: TN=88, FP=272, FN=518, TP=3960
  ROC AUC: 0.717 [0.692, 0.741]
  PR AUC: 0.967 [0.962, 0.972]
  Accuracy: 0.837 [0.826, 0.847]
  Balanced accuracy: 0.564 [0.542, 0.587]
  Precision: 0.936 [0.928, 0.943]
  Recall: 0.884 [0.875, 0.894]
  F1: 0.909 [0.903, 0.916]
  MCC: 0.102 [0.067, 0.140]
  Cohen's kappa: 0.098 [0.064, 0.133]
  Brier score: 0.119 [0.114, 0.125]
  R2: -0.528 [-0.602, -0.459]
  RMSE: 1.573 [1.549, 1.596]
  MAE: 1.330 [1.306, 1.353]
  Spearman rho: 0.324 [0.297, 0.348]
  Pearson r: 0.326 [0.302, 0.350]
      AD tier    n  ROC AUC      MCC        R2     RMSE  Spearman
    In-domain 4749 0.716385 0.108294 -0.534545 1.575517  0.321684
   Borderline   78 0.726627 0.000000 -0.797129 1.439306  0.325964
Out-of-domain   11 0.400000 0.000000 -0.321110 1.494003 -0.265452
        Source    n  Active %  ROC AUC       MCC  Spearman
ChEMBL_refresh 1441 90.562110 0.806857  0.244572  0.522002
     BindingDB 4197 91.946629 0.703128  0.091163  0.2971

============================================================================
SECTION 6 — Figures (PNG + SVG)
============================================================================


In [12]:
print("=" * 60)
print("SECTION 6 — Figures")
print("=" * 60)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Liberation Sans", "Arimo", "DejaVu Sans"]
plt.rcParams["svg.fonttype"] = "none"
TEAL, SLATE, AMBER, RED = "#0D9488", "#1E293B", "#D97706", "#DC2626"
FIGS = f"{OUT_DIR}/figures"
os.makedirs(FIGS, exist_ok=True)

def save(fig, name):
    fig.savefig(f"{FIGS}/{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{FIGS}/{name}.svg", bbox_inches="tight")
    plt.close(fig)

# ROC + PR
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fpr, tpr, _ = roc_curve(y_true, y_proba)
axes[0].plot(fpr, tpr, color=TEAL, lw=2, label=f"ROC AUC = {clf_rows[0]['Value']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="#94A3B8", lw=1, label="Random")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title("External ROC - classifier"); axes[0].legend(loc="lower right")
prec, rec, _ = precision_recall_curve(y_true, y_proba)
axes[1].plot(rec, prec, color=SLATE, lw=2, label=f"PR AUC = {clf_rows[1]['Value']:.3f}")
axes[1].axhline(y_true.mean(), ls="--", color="#94A3B8", lw=1,
                label=f"Prevalence = {y_true.mean():.2f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("External precision-recall - classifier"); axes[1].legend(loc="lower left")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
save(fig, "roc_pr_curves")

# Calibration
fig, ax = plt.subplots(figsize=(5.5, 5))
pt, pp = calibration_curve(y_true, y_proba, n_bins=10, strategy="quantile")
ax.plot([0, 1], [0, 1], "--", color="#94A3B8", lw=1, label="Perfectly calibrated")
ax.plot(pp, pt, "o-", color=TEAL, lw=2, label="Frozen classifier")
ax.set_xlabel("Mean predicted probability (active)")
ax.set_ylabel("Fraction actually active")
ax.set_title("External calibration - classifier"); ax.legend(loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
save(fig, "calibration_curve")

# Predicted vs experimental, colored by AD tier
tier_colors = {"In-domain": TEAL, "Borderline": AMBER, "Out-of-domain": RED}
fig, ax = plt.subplots(figsize=(6.5, 6))
for tier, color in tier_colors.items():
    sub = ext[ext["ad_tier"] == tier]
    ax.scatter(sub["pIC50"], sub["reg_pred_pIC50"], s=14, alpha=0.45, c=color,
               edgecolors="none", label=f"{tier} (n={len(sub)})")
lims = [min(y_exp.min(), y_reg.min()) - 0.3, max(y_exp.max(), y_reg.max()) + 0.3]
ax.plot(lims, lims, "--", color="#94A3B8", lw=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Experimental pIC50"); ax.set_ylabel("Predicted pIC50")
ax.set_title("External predicted vs experimental - regressor")
ax.legend(loc="upper left", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
save(fig, "pred_vs_exp_pIC50")

# Residuals
resid = y_reg - y_exp
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(y_exp, resid, s=14, alpha=0.4, c=SLATE, edgecolors="none")
ax.axhline(0, color=TEAL, lw=1.5)
ax.set_xlabel("Experimental pIC50"); ax.set_ylabel("Residual (predicted - experimental)")
ax.set_title("External residuals - regressor")
ax.spines[["top", "right"]].set_visible(False)
save(fig, "residuals")

# Source + AD composition
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
src_counts = ext["sources"].str.split(";").explode().value_counts()
axes[0].bar(src_counts.index, src_counts.values, color=[TEAL, SLATE, AMBER][:len(src_counts)])
for i, v in enumerate(src_counts.values):
    axes[0].text(i, v, str(v), ha="center", va="bottom", fontsize=9)
axes[0].set_title("External compounds per source"); axes[0].set_ylabel("Compounds")
axes[0].spines[["top", "right"]].set_visible(False)
tier_counts = ext["ad_tier"].value_counts()
axes[1].bar(tier_counts.index, tier_counts.values,
            color=[tier_colors[t] for t in tier_counts.index])
for i, v in enumerate(tier_counts.values):
    axes[1].text(i, v, str(v), ha="center", va="bottom", fontsize=9)
axes[1].set_title("Applicability domain distribution"); axes[1].set_ylabel("Compounds")
axes[1].spines[["top", "right"]].set_visible(False)
save(fig, "source_and_ad_composition")

print(f"Figures saved to {FIGS}")



SECTION 6 — Figures


Figures saved to /content/ppred_output/figures


============================================================================
SECTION 7 — Package outputs (download zip, optional Google Drive)
============================================================================


In [14]:
print("=" * 60)
print("SECTION 7 — Outputs")
print("=" * 60)
subprocess.run(["zip", "-r", "/content/ppred_external_validation_results.zip", OUT_DIR], check=False)
print(f"All outputs saved to: {OUT_DIR}")
print("Download: /content/ppred_external_validation_results.zip")
# print("""


SECTION 7 — Outputs
All outputs saved to: /content/ppred_output
Download: /content/ppred_external_validation_results.zip
